In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
from tqdm import tqdm
import statsmodels.api as sm

In [ ]:
with open("characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    # 计算残差
    model = LinearRegression(fit_intercept=False)
    model.fit(X_train, y_train)
    residuals = np.abs(model.predict(X_train) - y_train)

    # 计算残差的99.9%分位数
    epsilon = np.percentile(residuals, 99.9)

    # 根据计算得到的epsilon值重新拟合模型
    model = SGDRegressor(fit_intercept=False, loss='huber', epsilon=epsilon)
    model.fit(X_train, y_train)
   
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# 将 'Date' 列转换为日期时间类型
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
selected_features = ['mvel1']

# 存储原始的 R2 总和
original_r2 = -0.13811197839876943

# 存储每个特征设为 0 时的 R2 差值
r2_differences = {}

for feature in selected_features:
    pred_y_list = []
    true_y_list = []

# 循环处理每年的数据
    for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    
        start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
        end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

        start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
        end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)

        start_date_test = pd.Timestamp(year=year, month=1, day=1)
        end_date_test = pd.Timestamp(year=year, month=12, day=31)

        print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
        print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

        print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
        print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')

        print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
        print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

        train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
        test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

        X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values
        y_train = train_data['RET'].values
        X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
        y_test = test_data['RET'].values
   
        # 复制特征数据
        X_train_copy = X_train.copy()
        X_test_copy = X_test.copy()

        # 将当前特征设为 0
        X_train_copy[:, selected_features.index(feature)] = 0
        X_test_copy[:, selected_features.index(feature)] = 0

        model_zero= SGDRegressor(fit_intercept=False, loss='huber', epsilon=epsilon)
        model_zero.fit(X_train_copy, y_train)

        
        #进行预测
        y_test_pred_zero = model_zero.predict(X_test_copy)

        pred_y_list.extend(y_test_pred_zero)
        true_y_list.extend(y_test)
        

        # 计算新的 R2
    new_r2 = R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100

        # 计算 R2 的差值
    r2_difference = original_r2 - new_r2

        # 存储差值
    r2_differences[f'Feature_{feature}'] = r2_difference

# 打印每个特征设为 0 时的 R2 差值
print("每个特征设为 0 时的 R2 差值:")
for feature, difference in r2_differences.items():
    print(f"{feature}: {difference}")